In [40]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gradio as gr
from PIL import Image
import numpy as np 

In [43]:
train_datagen = ImageDataGenerator(
    rotation_range=20,      
    width_shift_range=0.1,  
    height_shift_range=0.1, 
    horizontal_flip=True,   
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = datagen.flow_from_directory("train", target_size=(64, 64),
                                     batch_size=32, class_mode="binary",
                                     subset="training")

val   = datagen.flow_from_directory("train", target_size=(64, 64),
                                     batch_size=32, class_mode="binary",
                                     subset="validation")

Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [44]:
base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(64, 64, 3))
base.trainable = False

model = tf.keras.Sequential([base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

C:\Users\damax\AppData\Local\Temp\ipykernel_6880\3964365727.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(64, 64, 3))


In [45]:
model.fit(train, validation_data=val, epochs=10) 

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 215s 84ms/step - accuracy: 0.8043 - loss: 0.4270 - val_accuracy: 0.8298 - val_loss: 0.3780
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 214s 86ms/step - accuracy: 0.8281 - loss: 0.3844 - val_accuracy: 0.8416 - val_loss: 0.3613
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 211s 84ms/step - accuracy: 0.8344 - loss: 0.3715 - val_accuracy: 0.8468 - val_loss: 0.3502
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 207s 83ms/step - accuracy: 0.8395 - loss: 0.3627 - val_accuracy: 0.8474 - val_loss: 0.3478
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 207s 83ms/step - accuracy: 0.8421 - loss: 0.3577 - val_accuracy: 0.8469 - val_loss: 0.3468
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 207s 83ms/step - accuracy: 0.8457 - loss: 0.3521 - val_accuracy: 0.8529 - val_loss: 0.3352
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 199s 80ms/step - accuracy: 0.8481 - loss: 0.3475 - val_accuracy: 0.8526 - val_loss: 0.3347
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 200s 80ms/step - accuracy: 

In [46]:
print(model.evaluate(val))

625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 63ms/step - accuracy: 0.8575 - loss: 0.3302
[0.33024996519088745, 0.857450008392334]


In [1]:
def predict(img):
    if img is None:
        return "⚠️ Please upload an image."
    img_resized = img.resize((64, 64))
    arr = np.array(img_resized) / 255.0
    arr = np.expand_dims(arr, axis=0)
    score = model.predict(arr)[0][0]
    label = "Real" if score > 0.1 else "AI-Generated"
    confidence = score if score > 0.1 else 1 - score
    emoji = "📷" if label == "Real" else "🤖"
    bar = "█" * int(confidence * 20) + "░" * (20 - int(confidence * 20))
    return f"""
{emoji}  {label}
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Confidence:  {confidence:.2%}
[{bar}]

AI Score:    {score:.4f}
Threshold:   0.5000
"""

css = """
body { font-family: 'Segoe UI', sans-serif; }
#title {
    text-align: center; color: #ffffff;
    font-size: 2em; font-weight: bold; padding: 20px;
    background: linear-gradient(135deg, #1a1a2e, #16213e);
    border-radius: 12px; margin-bottom: 10px;
}
#subtitle { text-align: center; color: #aaaaaa; margin-bottom: 20px; }
.gradio-container {
    background-color: #0f0f1a !important;
    max-width: 800px !important; margin: auto !important;
}
.upload-box {
    border: 2px dashed #4a4a8a !important;
    border-radius: 12px !important; background: #1a1a2e !important;
}
#result {
    background: #1a1a2e !important; border-radius: 12px !important;
    font-family: monospace !important; font-size: 1.1em !important;
    color: #00ff99 !important; padding: 20px !important;
    border: 1px solid #4a4a8a !important;
}
#detect-btn {
    background: linear-gradient(135deg, #4a4aff, #8a2be2) !important;
    border: none !important; border-radius: 8px !important;
    color: white !important; font-size: 1.1em !important;
    padding: 12px !important; cursor: pointer !important; width: 100% !important;
}
"""

with gr.Blocks() as interface:
    gr.HTML('<div id="title">🔍 AI Image Detector</div>')
    gr.HTML('<div id="subtitle">Upload any image to find out if it was taken by a camera or generated by AI</div>')
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image", elem_classes="upload-box")
            detect_btn = gr.Button("Detect", elem_id="detect-btn")
        with gr.Column():
            result_output = gr.Textbox(label="Result", lines=8,
                                       elem_id="result", interactive=False)
    detect_btn.click(fn=predict, inputs=image_input, outputs=result_output)

interface.launch(css=css, theme=gr.themes.Base())  

NameError: name 'gr' is not defined

In [37]:
print(train.class_indices)

{'fake': 0, 'real': 1}
